# Word (DOCX) Ingestion for RAG

This notebook shows two ingestion styles:
- Full document text extraction
- Element-level extraction for richer structure metadata

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import Docx2txtLoader, UnstructuredWordDocumentLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


PROJECT_ROOT = Path.cwd().resolve().parents[1]
DATA_DIR = PROJECT_ROOT / "data"
DOCX_PATH = DATA_DIR / "proposal.docx"

if not DOCX_PATH.exists():
    raise FileNotFoundError(f"Missing sample DOCX: {DOCX_PATH}")

print("DOCX path:", DOCX_PATH)

In [ ]:
docx_loader = Docx2txtLoader(str(DOCX_PATH))
docx_docs = docx_loader.load()

print("Docx2txtLoader")
print("documents:", len(docx_docs))
print("preview:", docx_docs[0].page_content[:180], "...")
print("metadata:", docx_docs[0].metadata)

In [ ]:
try:
    element_loader = UnstructuredWordDocumentLoader(str(DOCX_PATH), mode="elements")
    element_docs = element_loader.load()

    print("\nUnstructuredWordDocumentLoader")
    print("elements:", len(element_docs))
    print("sample categories:")
    for d in element_docs[:5]:
        print("-", d.metadata.get("category", "unknown"))
except Exception as exc:
    print("\nUnstructured loader is optional and may need extra dependencies.")
    print("error:", exc)
    element_docs = []

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=120,
)

source_docs = element_docs if element_docs else docx_docs
docx_chunks = splitter.split_documents(source_docs)

print("\nChunking")
print("chunks:", len(docx_chunks))
print("first chunk chars:", len(docx_chunks[0].page_content))
print("first chunk metadata keys:", list(docx_chunks[0].metadata.keys()))